# Build your own controller

This notebook teaches **Option A**: a Python controller drives the production
`EngineSession` (Rust kernel) in a simple loop:

1. Read belief + pipeline from `session.snapshot()`
2. Choose an order with your `controller.order(ctx)`
3. Advance physics + filter with `session.step(order_qty)`

We compare a **naive base-stock** baseline to a small **tabular Q-learning**
starter. Optionally we overlay one-step **rollout** from `session.act(policy="rollout")`.

Fixtures use `smoke_cool_shipments()` only (no Abdella parquet). This path does **not**
use `sim.episode.run_closed_loop_episode`.

## Setup

From the repo root with the Rust extension built:

```bash
uv sync --all-extras --python 3.11
uv run --python 3.11 maturin develop --release -m crates/voi_py/Cargo.toml
export BLUEBERRIES_VOI_BACKEND=rust
uv run jupyter lab notebooks/
```

Import the library helpers and confirm the Rust backend is available.

In [ ]:
%matplotlib inline

from __future__ import annotations

import os

os.environ.setdefault("BLUEBERRIES_VOI_BACKEND", "rust")

import matplotlib.pyplot as plt

from blueberries_voi.backend import rust_available, warn_fallback_once
from blueberries_voi.controller import (
    default_session_config,
    NaiveBaseStockController,
    run_controller_session,
    TabularQLearningController,
)
from blueberries_voi.simulator import EngineSession

plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": 0.3})

warn_fallback_once()
if not rust_available():
    raise RuntimeError("Build blueberries_voi._core and set BLUEBERRIES_VOI_BACKEND=rust")
print("Rust backend ready.")

### Simulation parameters

These knobs match the Studio demo budgets. Smaller particle / rollout counts run
faster in a notebook while keeping the same calendar and filter structure.

In [ ]:
# Random seed for EngineSession physics + filter RNG
SEED = 42

# Scored horizon (calendar days per controller run)
N_DAYS = 28

# Training episodes for tabular Q-learning
TRAIN_EPISODES = 8

# Particle filter size (lower = faster notebook)
N_PARTICLES = 64

# Rollout lookahead horizon inside Rust act() comparison
ROLLOUT_H = 5

# Rollout path count for act(policy="rollout")
N_ROLLOUT_PATHS = 2

# Case-radius for rollout candidate orders
CANDIDATE_CASE_RADIUS = 1

# Observation scenario preset (P1 = production default)
OBS_SCENARIO = "P1"

SESSION_CFG = default_session_config(
    n_particles=N_PARTICLES,
    H=ROLLOUT_H,
    n_rollout_paths=N_ROLLOUT_PATHS,
    candidate_case_radius=CANDIDATE_CASE_RADIUS,
    obs_scenario=OBS_SCENARIO,
)

### Controller parameters

The naive controller tops up to a fixed shelf target. The Q-learning starter explores
discrete order quantities on a coarse (weekday, on-hand bin) grid.

In [ ]:
# Naive base-stock target (units on shelf + pipeline)
BASE_STOCK_TARGET = 48

# Case size for naive rounding (matches ModelParams default)
CASE_SIZE = 8

# Discrete order actions for Q-learning (units, not cases)
QL_ACTIONS = [0, 8, 16, 24, 32]

# ε-greedy exploration probability during training
QL_EPSILON = 0.2

# Q-learning step size
QL_LEARNING_RATE = 0.15

# Discount factor (0 = maximize immediate day profit)
QL_DISCOUNT = 0.0

# On-hand cap for binning tabular states
QL_MAX_ON_HAND = 80

# Number of on-hand bins for tabular states
QL_ON_HAND_BINS = 8

## Train tabular Q-learning

Each training episode resets the session, runs `run_controller_session`, and calls
`observe` with **day profit** as the reward signal.

In [ ]:
ql = TabularQLearningController(
    QL_ACTIONS,
    epsilon=QL_EPSILON,
    learning_rate=QL_LEARNING_RATE,
    discount=QL_DISCOUNT,
    max_on_hand=QL_MAX_ON_HAND,
    on_hand_bins=QL_ON_HAND_BINS,
    seed=SEED,
)

train_profits: list[float] = []
for ep in range(TRAIN_EPISODES):
    session = EngineSession()
    session.init(SESSION_CFG, seed=SEED + ep)
    logs = run_controller_session(session, ql, N_DAYS)
    ep_profit = sum(log.day_profit for log in logs)
    train_profits.append(ep_profit)
    print(f"train episode {ep + 1}/{TRAIN_EPISODES}: profit={ep_profit:.1f}")

ql.epsilon = 0.0  # greedy evaluation after training

## Evaluate: naive base-stock vs trained Q-learning

We run one scored episode per controller with the same seed and config.

In [ ]:
def run_episode(controller, seed: int) -> list[float]:
    session = EngineSession()
    session.init(SESSION_CFG, seed=seed)
    logs = run_controller_session(session, controller, N_DAYS)
    return [log.day_profit for log in logs]

naive = NaiveBaseStockController(target_units=BASE_STOCK_TARGET, case_size=CASE_SIZE)
naive_daily = run_episode(naive, SEED)
ql_daily = run_episode(ql, SEED + 1000)

print(f"Naive total profit: {sum(naive_daily):.1f}")
print(f"Q-learning total profit: {sum(ql_daily):.1f}")

## Optional: rollout autopilot comparison

Rust `act(policy="rollout")` is **not** the training loop — we only use it here as a
reference curve for cumulative profit over the same horizon.

In [ ]:
from blueberries_voi.controller.session_loop import ControllerStepLog
from blueberries_voi.sim.profit import DEFAULT_PROFIT_COSTS

rollout_session = EngineSession()
rollout_session.init(SESSION_CFG, seed=SEED)
rollout_daily: list[float] = []
for _ in range(N_DAYS):
    delta = rollout_session.act(policy="rollout")
    log = ControllerStepLog.from_delta(delta, costs=DEFAULT_PROFIT_COSTS)
    rollout_daily.append(log.day_profit)

print(f"Rollout total profit: {sum(rollout_daily):.1f}")

## Cumulative profit comparison

In [ ]:
def cumulative(series: list[float]) -> list[float]:
    out: list[float] = []
    total = 0.0
    for x in series:
        total += x
        out.append(total)
    return out

fig, ax = plt.subplots()
days = list(range(1, len(naive_daily) + 1))
ax.plot(days, cumulative(naive_daily), label="Naive base-stock", color="#4C72B0")
ax.plot(days, cumulative(ql_daily), label="Tabular Q-learning", color="#55A868")
ax.plot(days, cumulative(rollout_daily), label="Rust rollout act()", color="#C44E52", ls="--")
ax.set_xlabel("Day")
ax.set_ylabel("Cumulative day profit")
ax.set_title("Controller comparison (smoke cool shipments)")
ax.legend()
fig.tight_layout()
plt.show()

## Next steps

- Swap `TabularQLearningController` for your own subclass of `ControllerTemplate`.
- Feed richer state from `ControllerContext.belief` (f-marginals per lot).
- Tune `BASE_STOCK_TARGET` or Q-learning actions against the same `run_controller_session` loop.
- Read ADR 0148 for why this path differs from `sim.episode.run_closed_loop_episode`.